# Initialization

In [0]:
import time
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_ingestion")

# Define Injestion Configuration

In [0]:
# List of source file ingestion configurations for CRM and ERP systems, specifying file path, and destination table
VOL_PATH = "/Volumes/workspace/bronze/raw_sources"
INGESTION_CONFIG = [
    {
        "path": VOL_PATH + "/source_crm/cust_info.csv",
        "dest_table": "crm_cust_info_raw"
    },
    {
        "path": VOL_PATH + "/source_crm/prd_info.csv",
        "dest_table": "crm_prd_info_raw"
    },
    {
        "path": VOL_PATH + "/source_crm/sales_details.csv",
        "dest_table": "crm_sales_details_raw"
    },
    {
        "path": VOL_PATH + "/source_erp/CUST_AZ12.csv",
        "dest_table": "erp_cust_az12_raw"
    },
    {
        "path": VOL_PATH + "/source_erp/LOC_A101.csv",
        "dest_table": "erp_loc_a101_raw"
    },
    {
        "path": VOL_PATH + "/source_erp/PX_CAT_G1V2.csv",
        "dest_table": "erp_px_cat_g1v2_raw"
    }
]

# Ingesting data into the Bronze tables

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA bronze")

In [0]:
for item in INGESTION_CONFIG:
    start = time.time()
    source = item['path']
    table = item['dest_table']

    try:
        logger.info(f"Processing {source}  ->  workspace.bronze.{table}")
        df = (
            spark.read
                .option("header", "true")
                .csv(source)
        )
        
        (
            df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(table)
        )

        duration = round(time.time() - start, 2)
        
        logger.info(f"SUCCESS: {table} | duration: {duration}s")

    except Exception as e:
        logger.error(f"FAILED: {source} | error: {e}")
    
            